# Backpropagation Through Time
:label:`sec_bptt`

If you completed the exercises in :numref:`sec_rnn-scratch`,
you would have seen that gradient clipping is vital 
for preventing the occasional massive gradients
from destabilizing training.
We hinted that the exploding gradients
stem from backpropagating across long sequences.
Before introducing a slew of modern RNN architectures,
let's take a closer look at how *backpropagation*
works in sequence models in mathematical detail.
Hopefully, this discussion will bring some precision 
to the notion of *vanishing* and *exploding* gradients.
If you recall our discussion of forward and backward 
propagation through computational graphs
when we introduced MLPs in :numref:`sec_backprop`,
then forward propagation in RNNs
should be relatively straightforward.
Applying backpropagation in RNNs 
is called *backpropagation through time* :cite:`Werbos.1990`.
This procedure requires us to expand (or unroll) 
the computational graph of an RNN
one time step at a time.
The unrolled RNN is essentially 
a feedforward neural network 
with the special property 
that the same parameters 
are repeated throughout the unrolled network,
appearing at each time step.
Then, just as in any feedforward neural network,
we can apply the chain rule, 
backpropagating gradients through the unrolled net.
The gradient with respect to each parameter
must be summed across all places 
that the parameter occurs in the unrolled net.
Handling such weight tying should be familiar 
from our chapters on convolutional neural networks.


Complications arise because sequences
can be rather long.
It is not unusual to work with text sequences
consisting of over a thousand tokens. 
Note that this poses problems both from 
a computational (too much memory)
and optimization (numerical instability)
standpoint. 
Input from the first step passes through
over 1000 matrix products before arriving at the output, 
and another 1000 matrix products 
are required to compute the gradient. 
We now analyze what can go wrong and 
how to address it in practice.


## Analysis of Gradients in RNNs
:label:`subsec_bptt_analysis`

We start with a simplified model of how an RNN works.
This model ignores details about the specifics 
of the hidden state and how it is updated.
The mathematical notation here
does not explicitly distinguish
scalars, vectors, and matrices.
We are just trying to develop some intuition.
In this simplified model,
we denote $h_t$ as the hidden state,
$x_t$ as input, and $o_t$ as output
at time step $t$.
Recall our discussions in
:numref:`subsec_rnn_w_hidden_states`
that the input and the hidden state
can be concatenated before being multiplied 
by one weight variable in the hidden layer.
Thus, we use $w_\textrm{h}$ and $w_\textrm{o}$ to indicate the weights 
of the hidden layer and the output layer, respectively.
As a result, the hidden states and outputs 
at each time step are

$$\begin{aligned}h_t &= f(x_t, h_{t-1}, w_\textrm{h}),\\o_t &= g(h_t, w_\textrm{o}),\end{aligned}$$
:eqlabel:`eq_bptt_ht_ot`

where $f$ and $g$ are transformations
of the hidden layer and the output layer, respectively.
Hence, we have a chain of values 
$\{\ldots, (x_{t-1}, h_{t-1}, o_{t-1}), (x_{t}, h_{t}, o_t), \ldots\}$ 
that depend on each other via recurrent computation.
The forward propagation is fairly straightforward.
All we need is to loop through the $(x_t, h_t, o_t)$ triples one time step at a time.
The discrepancy between output $o_t$ and the desired target $y_t$ 
is then evaluated by an objective function 
across all the $T$ time steps as

$$L(x_1, \ldots, x_T, y_1, \ldots, y_T, w_\textrm{h}, w_\textrm{o}) = \frac{1}{T}\sum_{t=1}^T l(y_t, o_t).$$



For backpropagation, matters are a bit trickier, 
especially when we compute the gradients 
with regard to the parameters $w_\textrm{h}$ of the objective function $L$. 
To be specific, by the chain rule,

$$\begin{aligned}\frac{\partial L}{\partial w_\textrm{h}}  & = \frac{1}{T}\sum_{t=1}^T \frac{\partial l(y_t, o_t)}{\partial w_\textrm{h}}  \\& = \frac{1}{T}\sum_{t=1}^T \frac{\partial l(y_t, o_t)}{\partial o_t} \frac{\partial g(h_t, w_\textrm{o})}{\partial h_t}  \frac{\partial h_t}{\partial w_\textrm{h}}.\end{aligned}$$
:eqlabel:`eq_bptt_partial_L_wh`

The first and the second factors of the
product in :eqref:`eq_bptt_partial_L_wh`
are easy to compute.
The third factor $\partial h_t/\partial w_\textrm{h}$ is where things get tricky, 
since we need to recurrently compute the effect of the parameter $w_\textrm{h}$ on $h_t$.
According to the recurrent computation
in :eqref:`eq_bptt_ht_ot`,
$h_t$ depends on both $h_{t-1}$ and $w_\textrm{h}$,
where computation of $h_{t-1}$
also depends on $w_\textrm{h}$.
Thus, evaluating the total derivate of $h_t$ 
with respect to $w_\textrm{h}$ using the chain rule yields

$$\frac{\partial h_t}{\partial w_\textrm{h}}= \frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial w_\textrm{h}} +\frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial h_{t-1}} \frac{\partial h_{t-1}}{\partial w_\textrm{h}}.$$
:eqlabel:`eq_bptt_partial_ht_wh_recur`


To derive the above gradient, assume that we have 
three sequences $\{a_{t}\},\{b_{t}\},\{c_{t}\}$ 
satisfying $a_{0}=0$ and $a_{t}=b_{t}+c_{t}a_{t-1}$ for $t=1, 2,\ldots$.
Then for $t\geq 1$, it is easy to show

$$a_{t}=b_{t}+\sum_{i=1}^{t-1}\left(\prod_{j=i+1}^{t}c_{j}\right)b_{i}.$$
:eqlabel:`eq_bptt_at`

By substituting $a_t$, $b_t$, and $c_t$ according to

$$\begin{aligned}a_t &= \frac{\partial h_t}{\partial w_\textrm{h}},\\
b_t &= \frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial w_\textrm{h}}, \\
c_t &= \frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial h_{t-1}},\end{aligned}$$

the gradient computation in :eqref:`eq_bptt_partial_ht_wh_recur` satisfies
$a_{t}=b_{t}+c_{t}a_{t-1}$.
Thus, per :eqref:`eq_bptt_at`, 
we can remove the recurrent computation 
in :eqref:`eq_bptt_partial_ht_wh_recur` with

$$\frac{\partial h_t}{\partial w_\textrm{h}}=\frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial w_\textrm{h}}+\sum_{i=1}^{t-1}\left(\prod_{j=i+1}^{t} \frac{\partial f(x_{j},h_{j-1},w_\textrm{h})}{\partial h_{j-1}} \right) \frac{\partial f(x_{i},h_{i-1},w_\textrm{h})}{\partial w_\textrm{h}}.$$
:eqlabel:`eq_bptt_partial_ht_wh_gen`

While we can use the chain rule to compute $\partial h_t/\partial w_\textrm{h}$ recursively, 
this chain can get very long whenever $t$ is large.
Let's discuss a number of strategies for dealing with this problem.

### Full Computation ### 

One idea might be to compute the full sum in :eqref:`eq_bptt_partial_ht_wh_gen`.
However, this is very slow and gradients can blow up,
since subtle changes in the initial conditions
can potentially affect the outcome a lot.
That is, we could see things similar to the butterfly effect,
where minimal changes in the initial conditions 
lead to disproportionate changes in the outcome.
This is generally undesirable.
After all, we are looking for robust estimators that generalize well. 
Hence this strategy is almost never used in practice.

### Truncating Time Steps###

Alternatively,
we can truncate the sum in
:eqref:`eq_bptt_partial_ht_wh_gen`
after $\tau$ steps. 
This is what we have been discussing so far. 
This leads to an *approximation* of the true gradient,
simply by terminating the sum at $\partial h_{t-\tau}/\partial w_\textrm{h}$. 
In practice this works quite well. 
It is what is commonly referred to as truncated 
backpropgation through time :cite:`Jaeger.2002`.
One of the consequences of this is that the model 
focuses primarily on short-term influence 
rather than long-term consequences. 
This is actually *desirable*, since it biases the estimate 
towards simpler and more stable models.


### Randomized Truncation ### 

Last, we can replace $\partial h_t/\partial w_\textrm{h}$
by a random variable which is correct in expectation 
but truncates the sequence.
This is achieved by using a sequence of $\xi_t$
with predefined $0 \leq \pi_t \leq 1$,
where $P(\xi_t = 0) = 1-\pi_t$ and 
$P(\xi_t = \pi_t^{-1}) = \pi_t$, thus $E[\xi_t] = 1$.
We use this to replace the gradient
$\partial h_t/\partial w_\textrm{h}$
in :eqref:`eq_bptt_partial_ht_wh_recur`
with

$$z_t= \frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial w_\textrm{h}} +\xi_t \frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial h_{t-1}} \frac{\partial h_{t-1}}{\partial w_\textrm{h}}.$$


It follows from the definition of $\xi_t$ 
that $E[z_t] = \partial h_t/\partial w_\textrm{h}$.
Whenever $\xi_t = 0$ the recurrent computation
terminates at that time step $t$.
This leads to a weighted sum of sequences of varying lengths,
where long sequences are rare but appropriately overweighted. 
This idea was proposed by 
:citet:`Tallec.Ollivier.2017`.

### Comparing Strategies

![Comparing strategies for computing gradients in RNNs. From top to bottom: randomized truncation, regular truncation, and full computation.](../img/truncated-bptt.svg)
:label:`fig_truncated_bptt`


:numref:`fig_truncated_bptt` illustrates the three strategies 
when analyzing the first few characters of *The Time Machine* 
using backpropagation through time for RNNs:

* The first row is the randomized truncation that partitions the text into segments of varying lengths.
* The second row is the regular truncation that breaks the text into subsequences of the same length. This is what we have been doing in RNN experiments.
* The third row is the full backpropagation through time that leads to a computationally infeasible expression.


Unfortunately, while appealing in theory, 
randomized truncation does not work 
much better than regular truncation, 
most likely due to a number of factors.
First, the effect of an observation
after a number of backpropagation steps 
into the past is quite sufficient 
to capture dependencies in practice. 
Second, the increased variance counteracts the fact 
that the gradient is more accurate with more steps. 
Third, we actually *want* models that have only 
a short range of interactions. 
Hence, regularly truncated backpropagation through time 
has a slight regularizing effect that can be desirable.

## Backpropagation Through Time in Detail

After discussing the general principle,
let's discuss backpropagation through time in detail.
In contrast to the analysis in :numref:`subsec_bptt_analysis`,
in the following we will show how to compute
the gradients of the objective function
with respect to all the decomposed model parameters.
To keep things simple, we consider 
an RNN without bias parameters,
whose activation function in the hidden layer
uses the identity mapping ($\phi(x)=x$).
For time step $t$, let the single example input 
and the target be $\mathbf{x}_t \in \mathbb{R}^d$ and $y_t$, respectively. 
The hidden state $\mathbf{h}_t \in \mathbb{R}^h$ 
and the output $\mathbf{o}_t \in \mathbb{R}^q$
are computed as

$$\begin{aligned}\mathbf{h}_t &= \mathbf{W}_\textrm{hx} \mathbf{x}_t + \mathbf{W}_\textrm{hh} \mathbf{h}_{t-1},\\
\mathbf{o}_t &= \mathbf{W}_\textrm{qh} \mathbf{h}_{t},\end{aligned}$$

where $\mathbf{W}_\textrm{hx} \in \mathbb{R}^{h \times d}$, $\mathbf{W}_\textrm{hh} \in \mathbb{R}^{h \times h}$, and
$\mathbf{W}_\textrm{qh} \in \mathbb{R}^{q \times h}$
are the weight parameters.
Denote by $l(\mathbf{o}_t, y_t)$
the loss at time step $t$. 
Our objective function,
the loss over $T$ time steps
from the beginning of the sequence is thus

$$L = \frac{1}{T} \sum_{t=1}^T l(\mathbf{o}_t, y_t).$$


In order to visualize the dependencies among
model variables and parameters during computation
of the RNN,
we can draw a computational graph for the model,
as shown in :numref:`fig_rnn_bptt`.
For example, the computation of the hidden states of time step 3,
$\mathbf{h}_3$, depends on the model parameters
$\mathbf{W}_\textrm{hx}$ and $\mathbf{W}_\textrm{hh}$,
the hidden state of the previous time step $\mathbf{h}_2$,
and the input of the current time step $\mathbf{x}_3$.

![Computational graph showing dependencies for an RNN model with three time steps. Boxes represent variables (not shaded) or parameters (shaded) and circles represent operators.](../img/rnn-bptt.svg)
:label:`fig_rnn_bptt`

As just mentioned, the model parameters in :numref:`fig_rnn_bptt` 
are $\mathbf{W}_\textrm{hx}$, $\mathbf{W}_\textrm{hh}$, and $\mathbf{W}_\textrm{qh}$. 
Generally, training this model requires 
gradient computation with respect to these parameters
$\partial L/\partial \mathbf{W}_\textrm{hx}$, $\partial L/\partial \mathbf{W}_\textrm{hh}$, and $\partial L/\partial \mathbf{W}_\textrm{qh}$.
According to the dependencies in :numref:`fig_rnn_bptt`,
we can traverse in the opposite direction of the arrows
to calculate and store the gradients in turn.
To flexibly express the multiplication of 
matrices, vectors, and scalars of different shapes
in the chain rule,
we continue to use the $\textrm{prod}$ operator 
as described in :numref:`sec_backprop`.


First of all, differentiating the objective function
with respect to the model output at any time step $t$
is fairly straightforward:

$$\frac{\partial L}{\partial \mathbf{o}_t} =  \frac{\partial l (\mathbf{o}_t, y_t)}{T \cdot \partial \mathbf{o}_t} \in \mathbb{R}^q.$$
:eqlabel:`eq_bptt_partial_L_ot`

Now we can calculate the gradient of the objective 
with respect to the parameter $\mathbf{W}_\textrm{qh}$
in the output layer:
$\partial L/\partial \mathbf{W}_\textrm{qh} \in \mathbb{R}^{q \times h}$. 
Based on :numref:`fig_rnn_bptt`, 
the objective $L$ depends on $\mathbf{W}_\textrm{qh}$ 
via $\mathbf{o}_1, \ldots, \mathbf{o}_T$. 
Using the chain rule yields

$$
\frac{\partial L}{\partial \mathbf{W}_\textrm{qh}}
= \sum_{t=1}^T \textrm{prod}\left(\frac{\partial L}{\partial \mathbf{o}_t}, \frac{\partial \mathbf{o}_t}{\partial \mathbf{W}_\textrm{qh}}\right)
= \sum_{t=1}^T \frac{\partial L}{\partial \mathbf{o}_t} \mathbf{h}_t^\top,
$$

where $\partial L/\partial \mathbf{o}_t$
is given by :eqref:`eq_bptt_partial_L_ot`.

Next, as shown in :numref:`fig_rnn_bptt`,
at the final time step $T$,
the objective function
$L$ depends on the hidden state $\mathbf{h}_T$ 
only via $\mathbf{o}_T$.
Therefore, we can easily find the gradient 
$\partial L/\partial \mathbf{h}_T \in \mathbb{R}^h$
using the chain rule:

$$\frac{\partial L}{\partial \mathbf{h}_T} = \textrm{prod}\left(\frac{\partial L}{\partial \mathbf{o}_T}, \frac{\partial \mathbf{o}_T}{\partial \mathbf{h}_T} \right) = \mathbf{W}_\textrm{qh}^\top \frac{\partial L}{\partial \mathbf{o}_T}.$$
:eqlabel:`eq_bptt_partial_L_hT_final_step`

It gets trickier for any time step $t < T$,
where the objective function $L$ depends on 
$\mathbf{h}_t$ via $\mathbf{h}_{t+1}$ and $\mathbf{o}_t$.
According to the chain rule,
the gradient of the hidden state
$\partial L/\partial \mathbf{h}_t \in \mathbb{R}^h$
at any time step $t < T$ can be recurrently computed as:


$$\frac{\partial L}{\partial \mathbf{h}_t} = \textrm{prod}\left(\frac{\partial L}{\partial \mathbf{h}_{t+1}}, \frac{\partial \mathbf{h}_{t+1}}{\partial \mathbf{h}_t} \right) + \textrm{prod}\left(\frac{\partial L}{\partial \mathbf{o}_t}, \frac{\partial \mathbf{o}_t}{\partial \mathbf{h}_t} \right) = \mathbf{W}_\textrm{hh}^\top \frac{\partial L}{\partial \mathbf{h}_{t+1}} + \mathbf{W}_\textrm{qh}^\top \frac{\partial L}{\partial \mathbf{o}_t}.$$
:eqlabel:`eq_bptt_partial_L_ht_recur`

For analysis, expanding the recurrent computation
for any time step $1 \leq t \leq T$ gives

$$\frac{\partial L}{\partial \mathbf{h}_t}= \sum_{i=t}^T {\left(\mathbf{W}_\textrm{hh}^\top\right)}^{T-i} \mathbf{W}_\textrm{qh}^\top \frac{\partial L}{\partial \mathbf{o}_{T+t-i}}.$$
:eqlabel:`eq_bptt_partial_L_ht`

We can see from :eqref:`eq_bptt_partial_L_ht` 
that this simple linear example already
exhibits some key problems of long sequence models:
it involves potentially very large powers of $\mathbf{W}_\textrm{hh}^\top$.
In it, eigenvalues smaller than 1 vanish
and eigenvalues larger than 1 diverge.
This is numerically unstable,
which manifests itself in the form of vanishing 
and exploding gradients.
One way to address this is to truncate the time steps
at a computationally convenient size 
as discussed in :numref:`subsec_bptt_analysis`. 
In practice, this truncation can also be effected 
by detaching the gradient after a given number of time steps.
Later on, we will see how more sophisticated sequence models 
such as long short-term memory can alleviate this further. 

Finally, :numref:`fig_rnn_bptt` shows 
that the objective function $L$ 
depends on model parameters $\mathbf{W}_\textrm{hx}$ and $\mathbf{W}_\textrm{hh}$
in the hidden layer via hidden states
$\mathbf{h}_1, \ldots, \mathbf{h}_T$.
To compute gradients with respect to such parameters
$\partial L / \partial \mathbf{W}_\textrm{hx} \in \mathbb{R}^{h \times d}$ and $\partial L / \partial \mathbf{W}_\textrm{hh} \in \mathbb{R}^{h \times h}$,
we apply the chain rule giving

$$
\begin{aligned}
\frac{\partial L}{\partial \mathbf{W}_\textrm{hx}}
&= \sum_{t=1}^T \textrm{prod}\left(\frac{\partial L}{\partial \mathbf{h}_t}, \frac{\partial \mathbf{h}_t}{\partial \mathbf{W}_\textrm{hx}}\right)
= \sum_{t=1}^T \frac{\partial L}{\partial \mathbf{h}_t} \mathbf{x}_t^\top,\\
\frac{\partial L}{\partial \mathbf{W}_\textrm{hh}}
&= \sum_{t=1}^T \textrm{prod}\left(\frac{\partial L}{\partial \mathbf{h}_t}, \frac{\partial \mathbf{h}_t}{\partial \mathbf{W}_\textrm{hh}}\right)
= \sum_{t=1}^T \frac{\partial L}{\partial \mathbf{h}_t} \mathbf{h}_{t-1}^\top,
\end{aligned}
$$

where $\partial L/\partial \mathbf{h}_t$
which is recurrently computed by
:eqref:`eq_bptt_partial_L_hT_final_step`
and :eqref:`eq_bptt_partial_L_ht_recur`
is the key quantity that affects the numerical stability.



Since backpropagation through time is the application of backpropagation in RNNs,
as we have explained in :numref:`sec_backprop`,
training RNNs alternates forward propagation with
backpropagation through time.
Moreover, backpropagation through time
computes and stores the above gradients in turn.
Specifically, stored intermediate values
are reused to avoid duplicate calculations,
such as storing $\partial L/\partial \mathbf{h}_t$
to be used in computation of both $\partial L / \partial \mathbf{W}_\textrm{hx}$ 
and $\partial L / \partial \mathbf{W}_\textrm{hh}$.


## Summary

Backpropagation through time is merely an application of backpropagation to sequence models with a hidden state.
Truncation, such as regular or randomized, is needed for computational convenience and numerical stability.
High powers of matrices can lead to divergent or vanishing eigenvalues. This manifests itself in the form of exploding or vanishing gradients.
For efficient computation, intermediate values are cached during backpropagation through time.



## Exercises

1. Assume that we have a symmetric matrix $\mathbf{M} \in \mathbb{R}^{n \times n}$ with eigenvalues $\lambda_i$ whose corresponding eigenvectors are $\mathbf{v}_i$ ($i = 1, \ldots, n$). Without loss of generality, assume that they are ordered in the order $|\lambda_i| \geq |\lambda_{i+1}|$. 
   1. Show that $\mathbf{M}^k$ has eigenvalues $\lambda_i^k$.
   1. Prove that for a random vector $\mathbf{x} \in \mathbb{R}^n$, with high probability $\mathbf{M}^k \mathbf{x}$ will be very much aligned with the eigenvector $\mathbf{v}_1$ 
of $\mathbf{M}$. Formalize this statement.
   1. What does the above result mean for gradients in RNNs?
1. Besides gradient clipping, can you think of any other methods to cope with gradient explosion in recurrent neural networks?

[Discussions](https://discuss.d2l.ai/t/334)


# 通过时间反向传播
:label:`sec_bptt`

如果你完成了 :numref:`sec_rnn-scratch` 中的练习，
就会发现梯度裁剪对于防止偶尔出现的巨大梯度破坏训练过程至关重要。
我们曾提到，这些爆炸梯度源于长时间序列上的反向传播。
在介绍一系列现代RNN架构之前，
让我们先通过数学细节来仔细看看序列模型中的*反向传播*是如何工作的。
希望这个讨论能对*梯度消失*和*梯度爆炸*的概念带来一些精确性。
如果你还记得我们在 :numref:`sec_backprop` 中介绍MLP时
讨论过的计算图的前向和反向传播，
那么RNN中的前向传播应该相对简单。
在RNN中应用反向传播被称为*通过时间反向传播* :cite:`Werbos.1990`。
这个过程需要我们逐步展开（或展开）
RNN的计算图。
展开后的RNN本质上是一个前馈神经网络，
具有特殊属性：
相同的参数在整个展开网络中重复出现，
出现在每个时间步。
然后，就像在任何前馈神经网络中一样，
我们可以应用链式法则，
通过展开的网络反向传播梯度。
每个参数的梯度必须对所有出现在展开网络中的该参数求和。
从我们关于卷积神经网络的章节中，
应该已经熟悉如何处理这种权重绑定。


问题在于序列可能相当长。
处理由超过一千个标记组成的文本序列并不罕见。
请注意，这会从计算（内存过多）
和优化（数值不稳定）
的角度带来问题。
来自第一步的输入在到达输出之前
要经过1000多个矩阵乘积，
还需要另外1000个矩阵乘积
来计算梯度。
我们现在分析可能出现的问题以及
如何在实践中解决它。


## RNN中的梯度分析
:label:`subsec_bptt_analysis`

我们从一个简化的RNN工作原理模型开始。
这个模型忽略了隐藏状态的具体细节
以及它是如何更新的。
这里的数学符号
没有明确区分
标量、向量和矩阵。
我们只是想发展一些直觉。
在这个简化模型中，
我们用$h_t$表示隐藏状态，
$x_t$表示输入，$o_t$表示输出
在时间步$t$。
回想一下我们在
:numref:`subsec_rnn_w_hidden_states`
中的讨论，
输入和隐藏状态
可以在隐藏层中与一个权重变量相乘之前
进行拼接。
因此，我们用$w_\textrm{h}$和$w_\textrm{o}$表示
隐藏层和输出层的权重。
因此，每个时间步的隐藏状态和输出
是

$$\begin{aligned}h_t &= f(x_t, h_{t-1}, w_\textrm{h}),\\o_t &= g(h_t, w_\textrm{o}),\end{aligned}$$
:eqlabel:`eq_bptt_ht_ot`

其中$f$和$g$分别是
隐藏层和输出层的变换。
因此，我们有一系列值
$\{\ldots, (x_{t-1}, h_{t-1}, o_{t-1}), (x_{t}, h_{t}, o_t), \ldots\}$ 
通过循环计算相互依赖。
前向传播相当简单。
我们只需要逐个时间步循环遍历$(x_t, h_t, o_t)$三元组。
输出$o_t$与目标$y_t$之间的差异
然后通过一个目标函数
在所有$T$个时间步上进行评估：

$$L(x_1, \ldots, x_T, y_1, \ldots, y_T, w_\textrm{h}, w_\textrm{o}) = \frac{1}{T}\sum_{t=1}^T l(y_t, o_t).$$


对于反向传播来说，情况有点复杂，
特别是当我们计算目标函数$L$关于参数$w_\textrm{h}$的梯度时。
具体来说，根据链式法则，

$$\begin{aligned}\frac{\partial L}{\partial w_\textrm{h}}  & = \frac{1}{T}\sum_{t=1}^T \frac{\partial l(y_t, o_t)}{\partial w_\textrm{h}}  \\& = \frac{1}{T}\sum_{t=1}^T \frac{\partial l(y_t, o_t)}{\partial o_t} \frac{\partial g(h_t, w_\textrm{o})}{\partial h_t}  \frac{\partial h_t}{\partial w_\textrm{h}}.\end{aligned}$$
:eqlabel:`eq_bptt_partial_L_wh`

:eqref:`eq_bptt_partial_L_wh`中乘积的第一个和第二个因子
很容易计算。
第三个因子$\partial h_t/\partial w_\textrm{h}$就比较棘手了，
因为我们需要递归地计算参数$w_\textrm{h}$对$h_t$的影响。
根据:eqref:`eq_bptt_ht_ot`中的循环计算，
$h_t$同时依赖于$h_{t-1}$和$w_\textrm{h}$，
而$h_{t-1}$的计算
也依赖于$w_\textrm{h}$。
因此，使用链式法则评估$h_t$
关于$w_\textrm{h}$的总导数得到

$$\frac{\partial h_t}{\partial w_\textrm{h}}= \frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial w_\textrm{h}} +\frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial h_{t-1}} \frac{\partial h_{t-1}}{\partial w_\textrm{h}}.$$
:eqlabel:`eq_bptt_partial_ht_wh_recur`


为了推导上述梯度，假设我们有
三个序列$\{a_{t}\},\{b_{t}\},\{c_{t}\}$ 
满足$a_{0}=0$且$a_{t}=b_{t}+c_{t}a_{t-1}$，其中$t=1, 2,\ldots$。
那么对于$t\geq 1$，容易证明

$$a_{t}=b_{t}+\sum_{i=1}^{t-1}\left(\prod_{j=i+1}^{t}c_{j}\right)b_{i}.$$
:eqlabel:`eq_bptt_at`

通过将$a_t$、$b_t$和$c_t$替换为

$$\begin{aligned}a_t &= \frac{\partial h_t}{\partial w_\textrm{h}},\\
b_t &= \frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial w_\textrm{h}}, \\
c_t &= \frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial h_{t-1}},\end{aligned}$$

:eqref:`eq_bptt_partial_ht_wh_recur`中的梯度计算满足
$a_{t}=b_{t}+c_{t}a_{t-1}$。
因此，根据:eqref:`eq_bptt_at`， 
我们可以用以下公式替代:eqref:`eq_bptt_partial_ht_wh_recur`中的递归计算：

$$\frac{\partial h_t}{\partial w_\textrm{h}}=\frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial w_\textrm{h}}+\sum_{i=1}^{t-1}\left(\prod_{j=i+1}^{t} \frac{\partial f(x_{j},h_{j-1},w_\textrm{h})}{\partial h_{j-1}} \right) \frac{\partial f(x_{i},h_{i-1},w_\textrm{h})}{\partial w_\textrm{h}}.$$
:eqlabel:`eq_bptt_partial_ht_wh_gen`

虽然我们可以使用链式法则递归地计算$\partial h_t/\partial w_\textrm{h}$， 
但当$t$很大时，这个链会变得非常长。
让我们讨论几种处理这个问题的策略。

### 完整计算 ### 

一个想法可能是计算:eqref:`eq_bptt_partial_ht_wh_gen`中的完整和。
然而，这非常慢且梯度可能会爆炸，
因为初始条件的微小变化
可能会极大地影响结果。
也就是说，我们可能会看到类似蝴蝶效应的现象，
初始条件的微小变化
会导致结果的巨大变化。
这通常是不希望的。
毕竟，我们寻找的是能够很好泛化的稳健估计器。 
因此，这个策略在实践中几乎从不使用。

### 截断时间步###

或者，
我们可以在
:eqref:`eq_bptt_partial_ht_wh_gen`
中截断求和，
只考虑最近的$\tau$步。 
这就是我们目前讨论的内容。 
这导致对真实梯度的*近似*，
通过在$\partial h_{t-\tau}/\partial w_\textrm{h}$处终止求和。 
在实践中，这效果很好。 
这就是通常所说的截断
通过时间反向传播:cite:`Jaeger.2002`。
这样做的后果之一是模型
主要关注短期影响
而不是长期后果。 
这实际上是*可取的*，因为它使估计
偏向于更简单和更稳定的模型。


### 随机截断 ### 

最后，我们可以用随机变量替换$\partial h_t/\partial w_\textrm{h}$，
这个随机变量在期望上是正确的，
但会截断序列。
这是通过使用一系列$\xi_t$
和预定义的$0 \leq \pi_t \leq 1$实现的，
其中$P(\xi_t = 0) = 1-\pi_t$且 
$P(\xi_t = \pi_t^{-1}) = \pi_t$，因此$E[\xi_t] = 1$。
我们用这个替换:eqref:`eq_bptt_partial_ht_wh_recur`
中的梯度
$\partial h_t/\partial w_\textrm{h}$
为

$$z_t= \frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial w_\textrm{h}} +\xi_t \frac{\partial f(x_{t},h_{t-1},w_\textrm{h})}{\partial h_{t-1}} \frac{\partial h_{t-1}}{\partial w_\textrm{h}}.$$


根据$\xi_t$的定义，
$E[z_t] = \partial h_t/\partial w_\textrm{h}$。
当$\xi_t = 0$时，递归计算
在该时间步$t$终止。
这导致不同长度序列的加权和，
其中长序列很少但会被适当加权。 
这个想法由
:citet:`Tallec.Ollivier.2017`提出。

### 策略比较

![比较RNN中计算梯度的策略。从上到下：随机截断、常规截断和完整计算。](../img/truncated-bptt.svg)
:label:`fig_truncated_bptt`


:numref:`fig_truncated_bptt`说明了三种策略，
当使用通过时间反向传播分析《时间机器》的前几个字符时：

* 第一行是随机截断，将文本划分为不同长度的段。
* 第二行是常规截断，将文本划分为相同长度的子序列。这就是我们在RNN实验中所做的。
* 第三行是完全通过时间反向传播，导致计算上不可行的表达式。


不幸的是，虽然在理论上很有吸引力，
但随机截断并不比常规截断
效果好多少，
很可能是由于多种因素。
首先，经过多次反向传播步骤后，
观察结果的影响
在实践中已经足以捕捉依赖关系。 
其次，增加的方差抵消了
梯度随着步骤增加而更准确的事实。 
第三，我们实际上*希望*模型只有
短范围的交互。 
因此，常规截断的通过时间反向传播
具有轻微的规范化效果，这可能是可取的。

## 通过时间反向传播的细节

在讨论了基本原理之后，
让我们详细讨论通过时间反向传播。
与:numref:`subsec_bptt_analysis`中的分析相比，
下面我们将展示如何计算
目标函数关于所有分解模型参数的梯度。
为了简单起见，我们考虑
一个没有偏置参数的RNN，
其隐藏层中的激活函数
使用恒等映射($\phi(x)=x$)。
对于时间步$t$，设单个示例输入
和目标分别为$\mathbf{x}_t \in \mathbb{R}^d$和$y_t$。 
隐藏状态$\mathbf{h}_t \in \mathbb{R}^h$ 
和输出$\mathbf{o}_t \in \mathbb{R}^q$
计算如下：

$$\begin{aligned}\mathbf{h}_t &= \mathbf{W}_\textrm{hx} \mathbf{x}_t + \mathbf{W}_\textrm{hh} \mathbf{h}_{t-1},\\
\mathbf{o}_t &= \mathbf{W}_\textrm{qh} \mathbf{h}_{t},\end{aligned}$$

其中$\mathbf{W}_\textrm{hx} \in \mathbb{R}^{h \times d}$、$\mathbf{W}_\textrm{hh} \in \mathbb{R}^{h \times h}$和
$\mathbf{W}_\textrm{qh} \in \mathbb{R}^{q \times h}$
是权重参数。
用$l(\mathbf{o}_t, y_t)$
表示时间步$t$的损失。 
我们的目标函数，
从序列开始到$T$个时间步的损失
因此是

$$L = \frac{1}{T} \sum_{t=1}^T l(\mathbf{o}_t, y_t).$$


为了可视化模型变量和参数之间的依赖关系
在计算RNN时，
我们可以为模型绘制一个计算图，
如:numref:`fig_rnn_bptt`所示。
例如，时间步3的隐藏状态$\mathbf{h}_3$的计算，
依赖于模型参数
$\mathbf{W}_\textrm{hx}$和$\mathbf{W}_\textrm{hh}$，
前一个时间步的隐藏状态$\mathbf{h}_2$，
以及当前时间步的输入$\mathbf{x}_3$。

![显示具有三个时间步的RNN模型依赖关系的计算图。方框表示变量（未阴影）或参数（阴影），圆圈表示运算符。](../img/rnn-bptt.svg)
:label:`fig_rnn_bptt`

如前所述，:numref:`fig_rnn_bptt`中的模型参数
是$\mathbf{W}_\textrm{hx}$、$\mathbf{W}_\textrm{hh}$和$\mathbf{W}_\textrm{qh}$。 
通常，训练这个模型需要
计算关于这些参数的梯度
$\partial L/\partial \mathbf{W}_\textrm{hx}$、$\partial L/\partial \mathbf{W}_\textrm{hh}$和$\partial L/\partial \mathbf{W}_\textrm{qh}$。
根据:numref:`fig_rnn_bptt`中的依赖关系，
我们可以沿着箭头的相反方向遍历
依次计算和存储梯度。
为了灵活表达不同形状的
矩阵、向量和标量的乘法
在链式法则中，
我们继续使用$\textrm{prod}$运算符，
如:numref:`sec_backprop`中所述。


首先，对任何时间步$t$，
目标函数关于模型输出的微分
相当简单：

$$\frac{\partial L}{\partial \mathbf{o}_t} =  \frac{\partial l (\mathbf{o}_t, y_t)}{T \cdot \partial \mathbf{o}_t} \in \mathbb{R}^q.$$
:eqlabel:`eq_bptt_partial_L_ot`

现在我们可以计算目标函数
关于输出层参数$\mathbf{W}_\textrm{qh}$的梯度：
$\partial L/\partial \mathbf{W}_\textrm{qh} \in \mathbb{R}^{q \times h}$。 
基于:numref:`fig_rnn_bptt`， 
目标$L$通过$\mathbf{o}_1, \ldots, \mathbf{o}_T$ 
依赖于$\mathbf{W}_\textrm{qh}$。 
使用链式法则得到

$$
\frac{\partial L}{\partial \mathbf{W}_\textrm{qh}}
= \sum_{t=1}^T \textrm{prod}\left(\frac{\partial L}{\partial \mathbf{o}_t}, \frac{\partial \mathbf{o}_t}{\partial \mathbf{W}_\textrm{qh}}\right)
= \sum_{t=1}^T \frac{\partial L}{\partial \mathbf{o}_t} \mathbf{h}_t^\top,
$$

其中$\partial L/\partial \mathbf{o}_t$
由:eqref:`eq_bptt_partial_L_ot`给出。

接下来，如:numref:`fig_rnn_bptt`所示，
在最终时间步$T$，
目标函数
$L$仅通过$\mathbf{o}_T$
依赖于隐藏状态$\mathbf{h}_T$。
因此，我们可以轻松找到梯度
$\partial L/\partial \mathbf{h}_T \in \mathbb{R}^h$
使用链式法则：

$$\frac{\partial L}{\partial \mathbf{h}_T} = \textrm{prod}\left(\frac{\partial L}{\partial \mathbf{o}_T}, \frac{\partial \mathbf{o}_T}{\partial \mathbf{h}_T} \right) = \mathbf{W}_\textrm{qh}^\top \frac{\partial L}{\partial \mathbf{o}_T}.$$
:eqlabel:`eq_bptt_partial_L_hT_final_step`

对于任何时间步$t < T$，
情况就变得复杂了，
因为目标函数$L$通过
$\mathbf{h}_{t+1}$和$\mathbf{o}_t$
依赖于$\mathbf{h}_t$。
根据链式法则，
隐藏状态的梯度
$\partial L/\partial \mathbf{h}_t \in \mathbb{R}^h$
在任何时间步$t < T$可以递归计算为：


$$\frac{\partial L}{\partial \mathbf{h}_t} = \textrm{prod}\left(\frac{\partial L}{\partial \mathbf{h}_{t+1}}, \frac{\partial \mathbf{h}_{t+1}}{\partial \mathbf{h}_t} \right) + \textrm{prod}\left(\frac{\partial L}{\partial \mathbf{o}_t}, \frac{\partial \mathbf{o}_t}{\partial \mathbf{h}_t} \right) = \mathbf{W}_\textrm{hh}^\top \frac{\partial L}{\partial \mathbf{h}_{t+1}} + \mathbf{W}_\textrm{qh}^\top \frac{\partial L}{\partial \mathbf{o}_t}.$$
:eqlabel:`eq_bptt_partial_L_ht_recur`

为了分析，展开递归计算
对于任何时间步$1 \leq t \leq T$得到

$$\frac{\partial L}{\partial \mathbf{h}_t}= \sum_{i=t}^T {\left(\mathbf{W}_\textrm{hh}^\top\right)}^{T-i} \mathbf{W}_\textrm{qh}^\top \frac{\partial L}{\partial \mathbf{o}_{T+t-i}}.$$
:eqlabel:`eq_bptt_partial_L_ht`

从:eqref:`eq_bptt_partial_L_ht`可以看出，
这个简单的线性例子已经
展示了长序列模型的一些关键问题：
它涉及$\mathbf{W}_\textrm{hh}^\top$的潜在非常大的幂。
其中，小于1的特征值会消失，
大于1的特征值会发散。
这在数值上是不稳定的，
表现为梯度消失
和梯度爆炸。
解决这个问题的一种方法是在计算上方便的大小处
截断时间步，
如:numref:`subsec_bptt_analysis`中讨论的。 
在实践中，这种截断也可以通过
在给定数量的时间步后分离梯度来实现。
稍后，我们将看到更复杂的序列模型，
如长短期记忆，可以进一步缓解这个问题。 

最后，:numref:`fig_rnn_bptt`显示
目标函数$L$ 
通过隐藏状态
$\mathbf{h}_1, \ldots, \mathbf{h}_T$
依赖于模型参数$\mathbf{W}_\textrm{hx}$和$\mathbf{W}_\textrm{hh}$。
为了计算关于这些参数的梯度
$\partial L / \partial \mathbf{W}_\textrm{hx} \in \mathbb{R}^{h \times d}$和$\partial L / \partial \mathbf{W}_\textrm{hh} \in \mathbb{R}^{h \times h}$，
我们应用链式法则得到

$$
\begin{aligned}
\frac{\partial L}{\partial \mathbf{W}_\textrm{hx}}
&= \sum_{t=1}^T \textrm{prod}\left(\frac{\partial L}{\partial \mathbf{h}_t}, \frac{\partial \mathbf{h}_t}{\partial \mathbf{W}_\textrm{hx}}\right)
= \sum_{t=1}^T \frac{\partial L}{\partial \mathbf{h}_t} \mathbf{x}_t^\top,\\
\frac{\partial L}{\partial \mathbf{W}_\textrm{hh}}
&= \sum_{t=1}^T \textrm{prod}\left(\frac{\partial L}{\partial \mathbf{h}_t}, \frac{\partial \mathbf{h}_t}{\partial \mathbf{W}_\textrm{hh}}\right)
= \sum_{t=1}^T \frac{\partial L}{\partial \mathbf{h}_t} \mathbf{h}_{t-1}^\top,
\end{aligned}
$$

其中$\partial L/\partial \mathbf{h}_t$
由:eqref:`eq_bptt_partial_L_hT_final_step`
和:eqref:`eq_bptt_partial_L_ht_recur`
递归计算，
是影响数值稳定性的关键量。

1. 用户反馈显示需要继续翻译"Summary"和"Exercises"部分
2. 当前仍处于Ask模式，无法直接写入文件
3. 需要保持技术术语一致性，特别注意数学公式和专有名词的翻译
4. 练习部分包含数学证明题，需要准确翻译数学表达

## 小结

通过时间反向传播仅仅是反向传播在具有隐藏状态的序列模型中的应用。
为了计算方便和数值稳定性，需要截断策略，如常规截断或随机截断。
矩阵的高次幂可能导致特征值发散或消失。这表现为梯度爆炸或梯度消失。
为了高效计算，在通过时间反向传播过程中会缓存中间值。

## 练习

1. 假设我们有一个对称矩阵$\mathbf{M} \in \mathbb{R}^{n \times n}$，其特征值为$\lambda_i$，对应的特征向量为$\mathbf{v}_i$（$i = 1, \ldots, n$）。不失一般性，假设它们按$|\lambda_i| \geq |\lambda_{i+1}|$的顺序排列。
   1. 证明$\mathbf{M}^k$的特征值为$\lambda_i^k$。
   1. 证明对于随机向量$\mathbf{x} \in \mathbb{R}^n$，$\mathbf{M}^k \mathbf{x}$极有可能与$\mathbf{M}$的特征向量$\mathbf{v}_1$高度对齐。请形式化这一表述。
   1. 上述结果对RNN中的梯度意味着什么？
2. 除了梯度裁剪，你还能想到其他应对循环神经网络中梯度爆炸的方法吗？

[讨论区](https://discuss.d2l.ai/t/334)